In [15]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("hdb_resale.csv")
df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [16]:
df["remaining_lease_years"] = df["remaining_lease"].str.split(" ").str[0].astype(int)

In [17]:
storeys = df["storey_range"].str.split(" TO ", expand=True).astype(int)
df["floor_avg"] = (storeys[0] + storeys[1]) / 2

In [18]:
df["month_dt"] = pd.to_datetime(df["month"])
df["months_since_start"] = (df["month_dt"].dt.year - 2017) * 12 + df["month_dt"].dt.month

In [19]:
df_encoded = pd.get_dummies(df, columns=["town", "flat_type", "flat_model"], drop_first=True)

In [20]:
feature_cols = ["floor_area_sqm", "remaining_lease_years", "floor_avg", "months_since_start"] + [c for c in df_encoded.columns if c.startswith("town_") or c.startswith("flat_type_") or c.startswith("flat_model_")]
X = df_encoded[feature_cols]
y = df_encoded["resale_price"]

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)
print("Linear Regression error:", mean_absolute_error(y_test, lr_predictions))

Linear Regression error: 52760.87851391757


In [23]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
print("Random Forest error:", mean_absolute_error(y_test, rf_predictions))

Random Forest error: 26811.401815377176


In [27]:
def predict_price(town, flat_type, flat_model, floor_area_sqm, remaining_lease_years, floor_avg, months_since_start):
    input_data = pd.DataFrame(columns=X.columns)
    input_data.loc[0] = 0

    input_data.loc[0, "floor_area_sqm"] = floor_area_sqm
    input_data.loc[0, "remaining_lease_years"] = remaining_lease_years
    input_data.loc[0, "floor_avg"] = floor_avg
    input_data.loc[0, "months_since_start"] = months_since_start

    town_col = f"town_{town}"
    flat_type_col = f"flat_type_{flat_type}"
    flat_model_col = f"flat_model_{flat_model}"

    if town_col in input_data.columns:
        input_data.loc[0, town_col] = 1
    if flat_type_col in input_data.columns:
        input_data.loc[0, flat_type_col] = 1
    if flat_model_col in input_data.columns:
        input_data.loc[0, flat_model_col] = 1

    predicted = rf_model.predict(input_data)[0]
    return f"${round(predicted):,}"

In [28]:
def months_since_jan_2017(year, month):
    return (year - 2017) * 12 + month

In [32]:
print(predict_price("ANG MO KIO", "3 ROOM", "New Generation", 73, 50, 5, months_since_jan_2017(2026, 8)))

$434,234


In [40]:
print(predict_price("HOUGANG", "5 ROOM", "New Generation", 120, 60, 15, months_since_jan_2017(2026, 8)))

$778,804


In [45]:
print(predict_price("ANG MO KIO", "5 ROOM", "Model A", 112, 84, 15, months_since_jan_2017(2026, 8)))

$936,113
